<a href="https://colab.research.google.com/github/TinkerTechie/LLM_LEARNING/blob/main/input_targetpairs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
!pip3 install tiktoken

In [31]:
import importlib
import tiktoken

In [32]:
tokenizer = tiktoken.get_encoding("gpt2")

In [33]:
with open("the-verdict.txt" ,"r" ,encoding="utf-8") as f:
  raw_text = f.read()
enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [34]:
enc_sample = enc_text[50:]

In [35]:
context_size = 4

In [36]:
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]
print(x)
print(y)

[290, 4920, 2241, 287]
[4920, 2241, 287, 257]


In [37]:
for i in range(1,context_size+1):
  context = enc_sample[:i]
  desired = enc_sample[i]
  print(tokenizer.decode(context),"->",tokenizer.decode([desired]))

 and ->  established
 and established ->  himself
 and established himself ->  in
 and established himself in ->  a


**implement data loader **

In [38]:
from torch.utils.data import Dataset,DataLoader

In [39]:
class GPTDatasetV1(Dataset):
  def __init__(self,txt,tokenizer,max_length,stride):
    self.input_ids = []
    self.target_ids = []

    token_ids = tokenizer.encode(txt,allowed_special={"<|endoftext|>"})

    for i in range(0,len(token_ids) - max_length,stride):
      input_chunk = token_ids[i:i+max_length]
      target_chunk = token_ids[i+1:i+max_length+1]
      self.input_ids.append(torch.tensor(input_chunk))
      self.target_ids.append(torch.tensor(target_chunk))



  def __len__(self):
    return len(self.input_ids)

  def __getitem__(self, idx):
    return self.input_ids[idx], self.target_ids[idx]

In [40]:
def createDataloader(txt,batch_size=4,max_length=256,stride=128,shuffle=True,drop_last=True,num_workers=0):
  tokenizer = tiktoken.get_encoding("gpt2")
  dataset = GPTDatasetV1(txt,tokenizer,max_length,stride)
  dataloader = DataLoader(
      dataset,
      batch_size=batch_size,
      shuffle= shuffle,
      drop_last=drop_last
      ,num_workers=num_workers
  )
  return dataloader

In [41]:
with open("the-verdict.txt" ,"r" ,encoding="utf-8") as f:
  raw_text = f.read()

In [42]:
import torch
dataloader = createDataloader(raw_text,batch_size=1,max_length=4,stride=1,shuffle=False)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]
